In [49]:
import pandas as pd

In [50]:
df = pd.read_csv('book.csv')

In [51]:
df.shape

(10000, 8)

In [52]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0449219364,G Is for Gumshoe (Kinsey Millhone Mysteries (P...,Sue Grafton,1991,Fawcett Books,http://images.amazon.com/images/P/0449219364.0...,http://images.amazon.com/images/P/0449219364.0...,http://images.amazon.com/images/P/0449219364.0...
1,0670891959,Fragments: The Collected Wisdom of Heraclitus,Heraclitus,2001,Viking Books,http://images.amazon.com/images/P/0670891959.0...,http://images.amazon.com/images/P/0670891959.0...,http://images.amazon.com/images/P/0670891959.0...
2,0679025731,Flashmaps Boston (Flashmaps),Marcy S. Pritchard,1994,Fodor's Travel Publications,http://images.amazon.com/images/P/0679025731.0...,http://images.amazon.com/images/P/0679025731.0...,http://images.amazon.com/images/P/0679025731.0...
3,0694517798,The Passion Dream Book,Whitney Otto,1997,Harper Audio,http://images.amazon.com/images/P/0694517798.0...,http://images.amazon.com/images/P/0694517798.0...,http://images.amazon.com/images/P/0694517798.0...
4,1893162052,Eat Yourself Slim,Michel Montignac,1999,Erica House,http://images.amazon.com/images/P/1893162052.0...,http://images.amazon.com/images/P/1893162052.0...,http://images.amazon.com/images/P/1893162052.0...


In [53]:
df.isnull().sum()

ISBN                   0
Book-Title             0
Book-Author            0
Year-Of-Publication    0
Publisher              0
Image-URL-S            0
Image-URL-M            0
Image-URL-L            0
dtype: int64

In [54]:
df = df.drop_duplicates()

In [55]:
df = df.drop(['ISBN','Image-URL-S','Image-URL-M','Image-URL-L'],axis=1)

In [56]:
df.head()

,Book-Title,Book-Author,Year-Of-Publication,Publisher
0,G Is for Gumshoe (Kinsey Millhone Mysteries (P...,Sue Grafton,1991,Fawcett Books
1,Fragments: The Collected Wisdom of Heraclitus,Heraclitus,2001,Viking Books
2,Flashmaps Boston (Flashmaps),Marcy S. Pritchard,1994,Fodor's Travel Publications
3,The Passion Dream Book,Whitney Otto,1997,Harper Audio
4,Eat Yourself Slim,Michel Montignac,1999,Erica House


In [57]:
df['overview'] = df['Book-Title']+ ' ' + df['Book-Author'] + ' ' + df['Year-Of-Publication'].astype(str) + ' ' + df['Publisher']

In [58]:
df.head()

,Book-Title,Book-Author,Year-Of-Publication,Publisher,overview
0,G Is for Gumshoe (Kinsey Millhone Mysteries (P...,Sue Grafton,1991,Fawcett Books,G Is for Gumshoe (Kinsey Millhone Mysteries (P...
1,Fragments: The Collected Wisdom of Heraclitus,Heraclitus,2001,Viking Books,Fragments: The Collected Wisdom of Heraclitus ...
2,Flashmaps Boston (Flashmaps),Marcy S. Pritchard,1994,Fodor's Travel Publications,Flashmaps Boston (Flashmaps) Marcy S. Pritchar...
3,The Passion Dream Book,Whitney Otto,1997,Harper Audio,The Passion Dream Book Whitney Otto 1997 Harpe...
4,Eat Yourself Slim,Michel Montignac,1999,Erica House,Eat Yourself Slim Michel Montignac 1999 Erica ...


In [59]:
import re 

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z0-9\s]",'',text)
    text = re.sub(r"\s+"," ", text).strip()
    return text

df['overview'] = df['overview'].apply(clean_text)

In [60]:
df.head()

,Book-Title,Book-Author,Year-Of-Publication,Publisher,overview
0,G Is for Gumshoe (Kinsey Millhone Mysteries (P...,Sue Grafton,1991,Fawcett Books,g is for gumshoe kinsey millhone mysteries pap...
1,Fragments: The Collected Wisdom of Heraclitus,Heraclitus,2001,Viking Books,fragments the collected wisdom of heraclitus h...
2,Flashmaps Boston (Flashmaps),Marcy S. Pritchard,1994,Fodor's Travel Publications,flashmaps boston flashmaps marcy s pritchard 1...
3,The Passion Dream Book,Whitney Otto,1997,Harper Audio,the passion dream book whitney otto 1997 harpe...
4,Eat Yourself Slim,Michel Montignac,1999,Erica House,eat yourself slim michel montignac 1999 erica ...


In [61]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')
embedding = model.encode(
    df['overview'].tolist(),
    show_progress_bar=True
)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [62]:
embedding.shape

(10000, 384)

In [63]:
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embedding)

In [64]:
def recommended(book_title, n=5):

    idx = df[df['Book-Title'].str.lower() == book_title.lower()].index[0]

    scores = list(enumerate(similarity[idx]))

    scores = sorted(
        scores,
        key=lambda x: x[1],
        reverse=True
    )

    book_indices = [i[0] for i in scores[1:n+1]]

    return df.iloc[book_indices][
        ['Book-Title', 'Book-Author', 'Publisher']
    ]

In [65]:
recommended('Eat Yourself Slim')

,Book-Title,Book-Author,Publisher
6839,"The Slim-Fast Body, Mind, Life Makeover",Lauren Hutton,Harpercollins
5603,I'm Still Hungry: Finding Myself Through Thick...,Carnie Wilson,Hay House
6037,"IT'S NOT WHAT YOU'RE EATING,ITS WHAT'S EATING YOU",Janet Greeson,Pocket
1938,All Night Long (Avon Romance),Michelle Jerott,Avon
8145,Dangerous,Nora Roberts,Silhouette
